# 02 — Preprocessing: Chuẩn hoá & Phân từ ViHSD


## 1. Dependencies & Cấu hình

In [2]:
import re
import os
import unicodedata
import pandas as pd
import py_vncorenlp

model_dir = '/content/vncorenlp'
os.makedirs(model_dir, exist_ok=True)

if not os.path.exists(os.path.join(model_dir, 'models')):
    py_vncorenlp.download_model(save_dir=model_dir)

rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=model_dir)

### Mount drive & Hugging Face

In [3]:
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

print(f"Project root: {PROJECT_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Hate_Speech_Detection


In [4]:
from huggingface_hub import login, HfApi, whoami

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
user_info = whoami()
print(f"Logged in as: {user_info['name']}")

Logged in as: AnoraLee


In [13]:
train = pd.read_csv(RAW_DIR / "vihsd_train.csv")
dev   = pd.read_csv(RAW_DIR / "vihsd_dev.csv")
test  = pd.read_csv(RAW_DIR / "vihsd_test.csv")

print(f"train={len(train):,} | dev={len(dev):,} | test={len(test):,}")

train=24,048 | dev=2,672 | test=6,680


## 2. Chuẩn hoá Unicode (NFC)


In [21]:
def normalize_unicode(text: str) -> str:
    text = str(text).replace('_', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return unicodedata.normalize("NFC", text)

for df in (train, dev, test):
    df["text"] = df["text"].apply(normalize_unicode)

## 3. Teencode Normalization

In [22]:
TOXIC_TEENCODE_MAP = {
    r"\bko\b": "không", r"\bhok\b": "không",
    r"\bdc\b": "được", r"\bđc\b": "được",
    r"\bj\b": "gì", r"\bbt\b": "bình thường",
    r"\btrc\b": "trước", r"\bnhg\b": "nhưng", r"\bthg\b": "thằng",

    r"\bdm\b": "địt mẹ", r"\bđm\b": "địt mẹ", r"\bdkm\b": "địt con mẹ", r"\bđkm\b": "địt con mẹ",
    r"\bvkl\b": "vãi lồn", r"\bvcl\b": "vãi lồn", r"\bvl\b": "vãi lồn", r"\bkl\b": "cái lồn",
    r"\bcc\b": "cục cứt", r"\bcđm\b": "cộng đồng mạng", r"\bml\b": "mặt lồn", r"\bcc\b": "con cặc",
    r"\bđjt\b": "địt", r"\bdjt\b": "địt", r"\bdit\b": "địt",
    r"\bloz\b": "lồn", r"\blon\b": "lồn",
    r"\bcac\b": "cặc", r"\bcặk\b": "cặc", r"\bđb\b": "đầu buồi"
}

def normalize_teencode(text: str) -> str:
    for pattern, replacement in TOXIC_TEENCODE_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

for df in (train, dev, test):
    df["text"] = df["text"].apply(normalize_teencode)

print("Examples:", train["text"].iloc[0])

Examples: Em được làm fan cứng luôn rồi nè ❤️ reaction quá hay quá cute coi mấy giờ này quá hợp lí = ] ] ]


## 4. Word Segmentation


In [23]:
def segment_text(text: str) -> str:
    try:
        sentences = rdrsegmenter.word_segment(text)
        return " ".join(sentences)
    except Exception:
        return text

for name, df in [("train", train), ("dev", dev), ("test", test)]:
    df["text_raw"] = df["text"]
    df["text"] = df["text_raw"].apply(segment_text)
    print(f"[{name}] Examples: {df['text_raw'].iloc[0]!r} -> {df['text'].iloc[0]!r}")

[train] Examples: 'Em được làm fan cứng luôn rồi nè ❤️ reaction quá hay quá cute coi mấy giờ này quá hợp lí = ] ] ]' -> 'Em được làm fan cứng luôn rồi nè ❤️ reaction quá hay quá cute coi mấy giờ này quá hợp_lí = ] ] ]'
[dev] Examples: 'Coi cười xỉu' -> 'Coi cười xỉu'
[test] Examples: 'Đừng cố biện minh = ) ) ) ) choi lồn' -> 'Đừng cố biện_minh = ) ) ) ) choi lồn'


## 5. Chuẩn hoá key để so trùng


In [24]:
def text_key(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text)).strip().lower()
    return re.sub(r"\s+", " ", text)

for df in (train, dev, test):
    df["_key"] = df["text_raw"].map(text_key)

## 6. Loại bỏ Leakage & Nhãn xung đột GIỮA các Split


In [25]:
all_data = pd.concat([train, dev, test], ignore_index=True)
conflict_keys = set(
    all_data.groupby("_key")["label"].nunique()
    .loc[lambda count: count > 1].index
)

test_before, dev_before, train_before = len(test), len(dev), len(train)

test = test[~test["_key"].isin(conflict_keys)].drop_duplicates("_key")
test_keys = set(test["_key"])

dev = dev[~dev["_key"].isin(conflict_keys | test_keys)].drop_duplicates("_key")
dev_keys = set(dev["_key"])

train = train[~train["_key"].isin(conflict_keys | test_keys | dev_keys)].drop_duplicates("_key")

train = train.reset_index(drop=True)
dev = dev.reset_index(drop=True)
test = test.reset_index(drop=True)

print(f"Texts with conflicting labels : {len(conflict_keys)}")
print(f"Test:       {test_before:,} -> {len(test):,}")
print(f"Validation: {dev_before:,} -> {len(dev):,}")
print(f"Train:      {train_before:,} -> {len(train):,}")

Texts with conflicting labels : 2
Test:       6,472 -> 6,470
Validation: 2,528 -> 2,528
Train:      21,261 -> 21,246


## 7. Lưu kết quả


In [26]:
train = train.drop(columns="_key")
dev = dev.drop(columns="_key")
test = test.drop(columns="_key")

train.to_csv(PROCESSED_DIR / "train.csv", index=False, encoding="utf-8-sig")
dev.to_csv(PROCESSED_DIR / "dev.csv", index=False, encoding="utf-8-sig")
test.to_csv(PROCESSED_DIR / "test.csv", index=False, encoding="utf-8-sig")

print(f"Đã lưu vào {PROCESSED_DIR}/: train.csv ({len(train):,}), dev.csv ({len(dev):,}), test.csv ({len(test):,})")

Đã lưu vào /content/drive/MyDrive/Hate_Speech_Detection/data/processed/: train.csv (21,246), dev.csv (2,528), test.csv (6,470)
